# Chemprop User Guide

**Version:** 2.2.3

**Audience:** Scientists and HPC users training molecular property prediction models with command-line workflows

## TL;DR

Chemprop v2 is a rewritten command-line toolkit for molecular property prediction. In this HPC environment, use it via modules and the v2 subcommand interface (`chemprop train`, `chemprop predict`, `chemprop fingerprint`).

Quick start:

```bash
module load chemprop/2.2.3
chemprop --help
chemprop train --help
```

This notebook focuses on reproducible CLI usage, SLURM-ready workflows, and troubleshooting on shared systems.

## Table of Contents

- [TL;DR](#tldr)
- [PART I - Getting Started (Read Once)](#part-i---getting-started-read-once)
  - [TL;DR: 5-Minute QuickStart](#tldr-5-minute-quickstart)
  - [Step 1: Load the Module](#step-1-load-the-module)
  - [Step 2: Verify Setup](#step-2-verify-setup)
  - [Step 3: Run Your First Job](#step-3-run-your-first-job)
  - [v2 Feature Highlights](#v2-feature-highlights)
- [PART II - How to Use Chemprop v2 (Read Carefully Once)](#part-ii---how-to-use-chemprop-v2-read-carefully-once)
  - [What You Need to Know](#what-you-need-to-know)
  - [CPU vs GPU: When to Use Each](#cpu-vs-gpu-when-to-use-each)
  - [Keyboard Shortcuts / Options](#keyboard-shortcuts--options)
  - [Resource Requests](#resource-requests)
  - [v2 Experiment Pattern for HPC](#v2-experiment-pattern-for-hpc)
- [PART III - Workflows (Use As Needed)](#part-iii---workflows-use-as-needed)
  - [Workflow 1: Train a Regression Model](#workflow-1-train-a-regression-model)
  - [Workflow 1.5: Hyperparameter Search Campaign with hpopt](#workflow-15-hyperparameter-search-campaign-with-hpopt)
  - [Workflow 2: Predict from a Saved Model](#workflow-2-predict-from-a-saved-model)
  - [Workflow 3: Generate Molecular Fingerprints](#workflow-3-generate-molecular-fingerprints)
  - [Workflow 3.5: Embedding Quality and Shape Checks](#workflow-35-embedding-quality-and-shape-checks)
  - [Workflow 4: Scalable Batch Prediction with Sharded Inputs](#workflow-4-scalable-batch-prediction-with-sharded-inputs)
- [PART IV - Troubleshooting (Skim When Broken)](#part-iv---troubleshooting-skim-when-broken)
  - [Symptom-Based Lookup](#symptom-based-lookup)
  - [FAQ](#faq)
  - [External Resources](#external-resources)

## PART I - Getting Started (Read Once)

### TL;DR: 5-Minute QuickStart

```bash
module load chemprop/2.2.3
chemprop --help
chemprop train --help
```

### Step 1: Load the Module

In [ ]:
%%bash
module avail chemprop 2>&1 | head -40

**Expected output:**
```text
One or more Chemprop modules are listed (for example chemprop/2.2.3).
```

In [ ]:
%%bash
module load chemprop/2.2.3
module list 2>&1 | head -40

**Expected output:**
```text
The loaded module list includes chemprop/2.2.3.
```

### Step 2: Verify Setup

In [ ]:
%%bash
module load chemprop/2.2.3
chemprop --help 2>&1 | head -30

**Expected output:**
```text
Usage text for the main chemprop CLI appears, including subcommands such as train, predict, convert, fingerprint, and hpopt.
```

Success! The Chemprop v2 command-line interface is available in your environment.

### Step 3: Run Your First Job

In [ ]:
%%bash
module load chemprop/2.2.3
for cmd in train predict convert hpopt fingerprint; do
  echo "===== chemprop ${cmd} --help ====="
  chemprop ${cmd} --help > chemprop_${cmd}_help.txt 2>&1 || true
  head -20 chemprop_${cmd}_help.txt
  echo
done

**Expected output:**
```text
Help summaries are printed for each major Chemprop v2 subcommand, and full help text is saved to chemprop_*_help.txt files.
```

Your first job is complete: you verified all major Chemprop v2 entry points and captured full option lists.

### v2 Feature Highlights

This notebook emphasizes high-impact v2 capabilities for real HPC usage:

- Unified subcommand interface for consistent job templates across train, predict, fingerprint, and hpopt.
- Native hyperparameter search entry point for controlled experiment campaigns.
- Representation workflows using fingerprint outputs for downstream analytics.
- Scalable prediction patterns using sharded inputs and merged outputs.

The workflows below are organized around these capabilities.

## PART II - How to Use Chemprop v2 (Read Carefully Once)

### What You Need to Know

- Chemprop v2 uses a unified top-level CLI with subcommands (for example `chemprop train`).
- Core workflow remains train first, then predict or fingerprint from saved model artifacts.
- Input molecular data is typically CSV with a SMILES column plus targets for training.
- In this environment, use module commands for setup and CLI commands for execution.

| Platform | Status | Notes |
|---|---|---|
| CPU-only nodes | Supported | Good for small datasets and debugging |
| GPU nodes | Supported | Recommended for larger datasets and faster training |
| Login node | Limited use | Use for setup, help, and lightweight checks only |

### CPU vs GPU: When to Use Each

Use CPU when:
- validating data format
- testing command syntax
- running very small demos

Use GPU when:
- training larger models or datasets
- running iterative experiments
- time-to-result matters

Decision guide:

```text
Prototype or small data?
  yes -> CPU
  no  -> GPU
```

#### How Chemprop v2 Selects the Device

No explicit GPU flag is required. All compute subcommands (`train`, `predict`, `fingerprint`, `hpopt`) default to:

```text
--accelerator auto   (Lightning selects GPU if available, falls back to CPU)
--devices auto       (Lightning selects available devices)
```

Both flags are passed directly to PyTorch Lightning's `Trainer`. On a GPU node with a CUDA-enabled build, GPU is used automatically with no changes to the command.

| Goal | Flag to add |
|---|---|
| Force CPU even on a GPU node | `--accelerator cpu` |
| Target a specific GPU | `--accelerator gpu --devices 0` |
| Use multiple GPUs | `--accelerator gpu --devices 0,1` |

**Note for `hpopt`:** Ray Tune has a separate GPU flag that is `False` by default. Add `--raytune-use-gpu` explicitly when running tuning campaigns on GPU nodes:

```bash
chemprop hpopt \
  --data-path data.csv \
  --task-type regression \
  --output-dir hpopt_out \
  --raytune-use-gpu
```

### Keyboard Shortcuts / Options

Chemprop is non-interactive CLI software, so focus on command options instead of keyboard shortcuts.

Key command families in v2:
- `chemprop train`: model training
- `chemprop predict`: batch inference
- `chemprop fingerprint`: learned representation extraction
- `chemprop hpopt`: hyperparameter search
- `chemprop convert`: convert legacy checkpoints

To inspect full flags for any command:

```bash
module load chemprop/2.2.3
chemprop train --help
```

### Resource Requests

Chemprop training is compute-intensive. Use scheduler allocations for real training jobs.

Example SLURM template:

```bash
#!/bin/bash
#SBATCH --job-name=chemprop-v2-train
#SBATCH --partition=gpu
#SBATCH --gres=gpu:1
#SBATCH --cpus-per-task=8
#SBATCH --mem=32G
#SBATCH --time=04:00:00

module load chemprop/2.2.3
chemprop train --data-path data.csv --task-type regression --output-dir train_out
```

For help-only and metadata checks, no allocation is needed on most clusters.

## PART III - Workflows (Use As Needed)

**Disclaimer (sequential workflow behavior):**

- Run Workflow 1 first. It creates the synthetic input data and baseline training artifacts.
- Workflows 2 and 3 are sequential and depend on files produced by Workflow 1.
- All three workflows use the same working directory: `./workflow`.
- Each downstream workflow performs file-existence checks before submitting jobs and exits early with a clear message if prerequisites are missing.

### Workflow 1: Train a Regression Model

In [ ]:
%%bash
set -euo pipefail

module load chemprop/2.2.3

WORKDIR=./workflow
rm -rf "${WORKDIR}"
mkdir -p "${WORKDIR}"
cd "${WORKDIR}"

cat > chemprop_workflow1.sbatch <<'SBATCH'
#!/bin/bash
#SBATCH --job-name=chemprop-v2-wf1
#SBATCH --partition=build
#SBATCH --cpus-per-task=4
#SBATCH --mem=8G
#SBATCH --time=00:20:00
#SBATCH --output=chemprop_workflow1.out

set -euo pipefail
module load chemprop/2.2.3

echo "Creating synthetic regression dataset..."
cat > regression_train.csv << 'CSV'
smiles,target
CC,0.10
CCC,0.20
CCCC,0.30
CCCCC,0.40
CCCCCC,0.50
C=O,0.60
CC=O,0.70
CCC=O,0.80
CCO,0.90
CCCO,1.00
CCCCO,1.10
CCN,1.20
CCCN,1.30
CCCCN,1.40
c1ccccc1,1.50
c1ccncc1,1.60
c1ccoc1,1.70
CCCl,1.80
CCCCl,1.90
CCBr,2.00
CSV

echo "Training Chemprop v2 model..."
chemprop train \
  --data-path regression_train.csv \
  --task-type regression \
  --output-dir train_example \
  --epochs 3 \
  --split-sizes 0.8 0.1 0.1 \
  --num-workers 0 \
  > train.log 2>&1

test -f train_example/model_0/best.pt

find train_example -maxdepth 4 -type f | sort > output_manifest.txt

echo "Workflow 1 completed successfully."
SBATCH

sbatch chemprop_workflow1.sbatch

**Expected output:**
```text
The command runs successfully and Chemprop creates training artifacts inside ./workflow.

Chemprop-generated outputs:
- train_example/model_0/best.pt (binary; listing only)
- regression_train.csv

Readable file preview (representative lines):
- regression_train.csv:
  - smiles,target
  - CC,0.10
  - CCC,0.20
```

### Workflow 1.5: Hyperparameter Search Campaign with hpopt

Goal: run a small, scheduler-ready tuning campaign and capture outputs for reproducible model selection.

In [ ]:
%%bash
set -euo pipefail

module load chemprop/2.2.3

WORKDIR=./workflow
test -f ${WORKDIR}/regression_train.csv || { echo Missing dataset artifact from Workflow 1; exit 1; }

cd ${WORKDIR}

cat > chemprop_workflow15_hpopt.sbatch <<'SBATCH'
#!/bin/bash
#SBATCH --job-name=chemprop-v2-hpopt
#SBATCH --partition=build
#SBATCH --cpus-per-task=4
#SBATCH --mem=8G
#SBATCH --time=00:20:00
#SBATCH --output=chemprop_workflow15_hpopt.out

set -euo pipefail
module load chemprop/2.2.3

mkdir -p hpopt_campaign

chemprop hpopt --help > hpopt_help.txt 2>&1

# Fill in exact flags from hpopt_help.txt for your site and run policy.
# Example structure:
# chemprop hpopt \
#   --data-path regression_train.csv \
#   --task-type regression \
#   --output-dir hpopt_campaign

echo Captured hpopt options and prepared campaign directory.
echo Workflow 1.5 completed successfully.
SBATCH

sbatch chemprop_workflow15_hpopt.sbatch

**Expected output:**
```text
The workflow confirms dataset prerequisites, executes hpopt --help to document available hyperparameter options, and prepares the campaign directory structure.

Chemprop-generated outputs:
- hpopt_help.txt (captured help documentation)
- hpopt_campaign/ (empty directory ready for campaign run)

Readable file preview (representative lines of hpopt_help.txt):
- usage: chemprop hpopt [-h] [--data-path DATA_PATH] [--task-type TASK_TYPE]
- optional arguments:
-   --data-path DATA_PATH   Path to training data.csv
-   --task-type TASK_TYPE   [regression|classification|multiclass]
- ... (additional help entries)

Note: Edit hpopt command with your specific hyperparameter search space before uncommenting the full run. See Common Patterns section above for Ray Tune configuration.
```

### Workflow 2: Predict from a Saved Model

In [ ]:
%%bash
set -euo pipefail

module load chemprop/2.2.3

WORKDIR=./workflow
test -f "${WORKDIR}/train_example/model_0/best.pt" || { echo "Missing model artifact from Workflow 1"; exit 1; }
test -f "${WORKDIR}/regression_train.csv" || { echo "Missing dataset artifact from Workflow 1"; exit 1; }

cd "${WORKDIR}"

cat > chemprop_workflow2.sbatch <<'SBATCH'
#!/bin/bash
#SBATCH --job-name=chemprop-v2-wf2
#SBATCH --partition=build
#SBATCH --cpus-per-task=4
#SBATCH --mem=8G
#SBATCH --time=00:20:00
#SBATCH --output=chemprop_workflow2.out

set -euo pipefail
module load chemprop/2.2.3

test -f train_example/model_0/best.pt
test -f regression_train.csv

chemprop predict \
  --test-path regression_train.csv \
  --model-path train_example/model_0/best.pt \
  --preds-path preds_reg.csv \
  > predict.log 2>&1

if [ -f preds_reg.csv ]; then
  PRED_FILE=preds_reg.csv
elif [ -f preds_reg_0.csv ]; then
  PRED_FILE=preds_reg_0.csv
else
  echo "Prediction output file not found"
  exit 1
fi

test -s "${PRED_FILE}"
head -n 10 "${PRED_FILE}"
echo "Workflow 2 completed successfully."
SBATCH

sbatch chemprop_workflow2.sbatch

**Expected output:**
```text
The command checks for Workflow 1 artifacts first and exits with a clear message if they are missing.
If prerequisites exist, it runs successfully and Chemprop writes predictions.

Chemprop-generated outputs:
- preds_reg.csv (or preds_reg_0.csv, depending on output naming mode)

Readable file preview (representative lines):
- preds_reg.csv:
  - smiles,target,pred_0
  - CC,0.10,0.12
  - CCC,0.20,0.19

Note: prediction values vary by run and model state.
```

### Workflow 3: Generate Molecular Fingerprints

In [ ]:
%%bash
set -euo pipefail

module load chemprop/2.2.3

WORKDIR=./workflow
test -f "${WORKDIR}/train_example/model_0/best.pt" || { echo "Missing model artifact from Workflow 1"; exit 1; }
test -f "${WORKDIR}/regression_train.csv" || { echo "Missing dataset artifact from Workflow 1"; exit 1; }

cd "${WORKDIR}"

cat > chemprop_workflow3.sbatch <<'SBATCH'
#!/bin/bash
#SBATCH --job-name=chemprop-v2-wf3
#SBATCH --partition=build
#SBATCH --cpus-per-task=4
#SBATCH --mem=8G
#SBATCH --time=00:20:00
#SBATCH --output=chemprop_workflow3.out

set -euo pipefail
module load chemprop/2.2.3

test -f train_example/model_0/best.pt
test -f regression_train.csv

chemprop fingerprint \
  --test-path regression_train.csv \
  --model-path train_example/model_0/best.pt \
  --preds-path fingerprint.csv \
  --ffn-block-index -1 \
  > fingerprint.log 2>&1

if [ -f fingerprint.csv ]; then
  FP_FILE=fingerprint.csv
elif [ -f fingerprint_0.csv ]; then
  FP_FILE=fingerprint_0.csv
else
  echo "Fingerprint output file not found"
  exit 1
fi

test -s "${FP_FILE}"
head -n 5 "${FP_FILE}"
echo "Workflow 3 completed successfully."
SBATCH

sbatch chemprop_workflow3.sbatch 

**Expected output:**
```text
The command checks for Workflow 1 artifacts first and exits with a clear message if they are missing.
If prerequisites exist, it runs successfully and Chemprop writes fingerprints.

Chemprop-generated outputs:
- fingerprint.csv (or fingerprint_0.csv, depending on output naming mode)

Readable file preview (representative lines):
- fingerprint.csv:
  - smiles,fp_0,fp_1,fp_2,...
  - CC,0.12,-0.03,0.77,...
  - CCC,0.18,-0.10,0.81,...

Note: fingerprint dimensionality and numeric values depend on model configuration.
```

### Workflow 3.5: Embedding Quality and Shape Checks

Goal: turn fingerprint output into immediately actionable diagnostics before downstream ML.

In [ ]:
%%bash
set -euo pipefail

module load chemprop/2.2.3

WORKDIR=./workflow
if [ -f "${WORKDIR}/fingerprint.csv" ] || [ -f "${WORKDIR}/fingerprint_0.csv" ]; then
  echo "Fingerprint artifact found. Proceeding."
else
  echo "Fingerprint output not found. Run Workflow 3 first."
  exit 1
fi

cd "${WORKDIR}"

cat > chemprop_workflow35_embed.sbatch <<'SBATCH'
#!/bin/bash
#SBATCH --job-name=chemprop-v2-embed
#SBATCH --partition=build
#SBATCH --cpus-per-task=2
#SBATCH --mem=4G
#SBATCH --time=00:05:00
#SBATCH --output=chemprop_workflow35_embed.out

set -euo pipefail
module load chemprop/2.2.3

if [ -f fingerprint.csv ]; then
  FP_FILE=fingerprint.csv
elif [ -f fingerprint_0.csv ]; then
  FP_FILE=fingerprint_0.csv
else
  echo "Fingerprint output not found."
  exit 1
fi

ROWS=$(tail -n +2 "${FP_FILE}" | wc -l | tr -d ' ')
FP_DIMS=$(head -1 "${FP_FILE}" | tr ',' '\n' | grep -c '^fp_')
FP0_COL=$(head -1 "${FP_FILE}" | tr ',' '\n' | grep -n '^fp_0$' | cut -d: -f1)
FP0_MEAN=$(tail -n +2 "${FP_FILE}" | awk -F',' -v col="${FP0_COL}" '{sum+=$col; count++} END {if(count>0) printf "%.6f", sum/count}')

echo "file=${FP_FILE}"
echo "rows=${ROWS}"
echo "fingerprint_dimensions=${FP_DIMS}"
echo "fp_0_mean=${FP0_MEAN}"
echo "Workflow 3.5 completed successfully."
SBATCH

sbatch chemprop_workflow35_embed.sbatch

**Expected output:**
```text
The workflow reads the fingerprint CSV from Workflow 3, calculates embedding statistics, and displays a compact diagnostic report.

Sample output:
file=fingerprint.csv
rows=20
fingerprint_dimensions=300
fp_0_mean=0.125847

These values indicate: 20 molecules were fingerprinted, each with 300-dimensional representations, and the average value in the first embedding dimension is ~0.126. Use this as a quick sanity check before downstream ML.
```

### Workflow 4: Scalable Batch Prediction with Sharded Inputs

Goal: show production-style throughput by splitting inference data into shards and merging predictions.

In [ ]:
%%bash
set -euo pipefail

module load chemprop/2.2.3

WORKDIR=./workflow
test -f "${WORKDIR}/train_example/model_0/best.pt" || { echo "Missing model artifact from Workflow 1"; exit 1; }
test -f "${WORKDIR}/regression_train.csv" || { echo "Missing dataset artifact from Workflow 1"; exit 1; }

cd "${WORKDIR}"
mkdir -p shards shard_preds

cat > chemprop_workflow4_sharded.sbatch <<'SBATCH'
#!/bin/bash
#SBATCH --job-name=chemprop-v2-sharded
#SBATCH --partition=build
#SBATCH --cpus-per-task=4
#SBATCH --mem=8G
#SBATCH --time=00:20:00
#SBATCH --output=chemprop_workflow4_sharded.out

set -euo pipefail
module load chemprop/2.2.3

HEADER=$(head -1 regression_train.csv)
TOTAL=$(tail -n +2 regression_train.csv | wc -l)
CHUNK=$(( (TOTAL + 2) / 3 ))

tail -n +2 regression_train.csv | split -l "${CHUNK}" - shards/raw_

i=1
for raw in shards/raw_*; do
  out="shards/test_part_${i}.csv"
  echo "${HEADER}" > "${out}"
  cat "${raw}" >> "${out}"
  rm "${raw}"
  i=$(( i + 1 ))
done

for shard in shards/test_part_*.csv; do
  base=$(basename "${shard}" .csv)
  chemprop predict \
    --test-path "${shard}" \
    --model-path train_example/model_0/best.pt \
    --preds-path "shard_preds/${base}_preds.csv" > "shard_preds/${base}.log" 2>&1
done

FIRST_PRED=$(ls shard_preds/*_preds.csv 2>/dev/null | head -1 || true)
if [ -z "${FIRST_PRED}" ]; then
  echo "No shard prediction files were created."
  exit 1
fi

head -1 "${FIRST_PRED}" > preds_merged.csv
for f in shard_preds/*_preds.csv; do
  tail -n +2 "${f}" >> preds_merged.csv
done

test -s preds_merged.csv
head -n 10 preds_merged.csv
echo "Workflow 4 completed successfully."
SBATCH

sbatch chemprop_workflow4_sharded.sbatch

**Expected output:**
```text
The workflow splits the regression dataset into 3 shards, runs predictions on each, and merges results into a single file.

Chemprop-generated outputs:
- preds_merged.csv

Readable file preview (representative lines):
- preds_merged.csv:
  - smiles,target,pred_0
  - CC,0.10,0.11
  - CCC,0.20,0.21
  - CCCC,0.30,0.29
  - ... (17 more rows)

Note: Per-shard predictions are stored in shard_preds/ for inspection; merged file is ready for downstream analysis or validation.
```

## PART IV - Troubleshooting (Skim When Broken)

### Symptom-Based Lookup

| Symptom | Likely Cause | Fix |
|---|---|---|
| `chemprop: command not found` | Module not loaded | `module load chemprop/2.2.3` |
| `hpopt` exits early | Required options not fully specified | Capture options with `chemprop hpopt --help` and start from a minimal template |
| Prediction output filename differs from expected | Output naming mode writes indexed files | Check for both base and indexed output names and proceed with detected file |
| Fingerprint file generated but hard to validate | No post-check step | Run embedding quality cell to confirm row count and dimension shape |
| Throughput too slow on large inference set | Single-file serial prediction | Use shard workflow and merge outputs after parallel runs |
| CUDA not detected | Running on CPU node or missing GPU allocation | Submit to GPU partition and request a GPU resource |
| CSV parsing or schema error | Invalid header or missing columns | Validate schema and specify smiles and target columns explicitly |

### FAQ

**Q: Can I still run `chemprop_train` in v2?**

A: Use the v2 interface (`chemprop train`, `chemprop predict`, `chemprop fingerprint`). The old v1-style entry-point names are not the primary interface in v2.

**Q: Which command should I start with?**

A: Start with `chemprop --help`, then `chemprop train --help`, then run a short training job on a small dataset.

**Q: How do I convert older checkpoints?**

A: Use `chemprop convert --help` and choose the appropriate conversion mode.

### External Resources

- Official repository: https://github.com/chemprop/chemprop
- Main documentation: https://chemprop.readthedocs.io/en/main/
- CLI tutorials: https://chemprop.readthedocs.io/en/main/tutorial/cli/index.html
- CLI reference: https://chemprop.readthedocs.io/en/main/cmd.html
- Publication (v2): https://doi.org/10.1021/acs.jcim.5c02332